In [6]:
# run classical ML model on ref-alt matrix

In [24]:
import os
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV, RidgeCV, LogisticRegressionCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.preprocessing import LabelEncoder

In [25]:
# ---------------------- Utility Functions ----------------------

def load_feature_matrix_and_labels(gene_name):
    X = np.load(f"./data/feature_matrix_labels/{gene_name}_feature_matrix.npy")
    y = np.load(f"./data/feature_matrix_labels/{gene_name}_labels.npy", allow_pickle=True)
    return X, y

def encode_labels(y):
    le = LabelEncoder()
    return le.fit_transform(y)

def run_models(gene_name):
    print(f"\n=== Running models for {gene_name} ===")
    X, y = load_feature_matrix_and_labels(gene_name)
    y_encoded = encode_labels(y)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
    )

    results = {}

    # ----- Lasso -----
    lasso = LassoCV(max_iter=10000, cv=5, random_state=42, alphas=[0.001, 0.01, 0.1, 1, 10, 100])
    lasso.fit(X_train, y_train)
    results['lasso'] = {
        'r2': lasso.score(X_test, y_test),
        'auc': roc_auc_score(y_test, lasso.predict(X_test)),
        'mse': mean_squared_error(y_test, lasso.predict(X_test))
    }

    # ----- Ridge -----
    ridge = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10])
    ridge.fit(X_train, y_train)
    results['ridge'] = {
        'r2': ridge.score(X_test, y_test),
        'auc': roc_auc_score(y_test, ridge.predict(X_test)),
        'mse': mean_squared_error(y_test, ridge.predict(X_test))
    }

    # ----- Logistic Regression -----
    log_model = LogisticRegressionCV(
        cv=3, scoring="roc_auc", max_iter=5000,
        Cs=[1e-4, 1e-3, 1e-2, 0.1, 1, 10, 100],
        class_weight="balanced"
    )
    log_model.fit(X_train, y_train)
    results['logreg'] = {
        'r2': log_model.score(X_test, y_test),
        'auc': roc_auc_score(y_test, log_model.predict(X_test)),
        'mse': mean_squared_error(y_test, log_model.predict(X_test))
    }

    # ----- Output -----
    for model_name, metrics in results.items():
        print(f"\n[{model_name.upper()}] Results for {gene_name}:")
        for metric, val in metrics.items():
            print(f"{metric}: {val:.4f}")

    return results


In [26]:
ref_seqs=pd.read_csv('./data/catalog/protein_sequences.csv')
gene_list=['rpoB','rpsL','tlyA','pncA','eis','gid','katG','inhA','embA','embB', 'embC','gyrB', 'gyrA', 'ethA', 'ethR']

In [27]:

all_results = {}
for gene in gene_list:
    try:
        all_results[gene] = run_models(gene)
    except Exception as e:
        print(f"Error processing {gene}: {e}")



=== Running models for rpoB ===

[LASSO] Results for rpoB:
r2: 0.8184
auc: 0.9583
mse: 0.0321

[RIDGE] Results for rpoB:
r2: 0.8346
auc: 0.9628
mse: 0.0292

[LOGREG] Results for rpoB:
r2: 0.9626
auc: 0.9569
mse: 0.0311

=== Running models for rpsL ===

[LASSO] Results for rpsL:
r2: 0.4353
auc: 0.7739
mse: 0.1205

[RIDGE] Results for rpsL:
r2: 0.4354
auc: 0.7739
mse: 0.1205

[LOGREG] Results for rpsL:
r2: 0.7739
auc: 0.7211
mse: 0.1734

=== Running models for tlyA ===

[LASSO] Results for tlyA:
r2: -0.0000
auc: 0.5000
mse: 0.1173

[RIDGE] Results for tlyA:
r2: 0.0052
auc: 0.5024
mse: 0.1166

[LOGREG] Results for tlyA:
r2: 0.5024
auc: 0.5052
mse: 0.1343

=== Running models for pncA ===

[LASSO] Results for pncA:
r2: 0.1448
auc: 0.6239
mse: 0.0949

[RIDGE] Results for pncA:
r2: 0.4324
auc: 0.8037
mse: 0.0630

[LOGREG] Results for pncA:
r2: 0.8352
auc: 0.6363
mse: 0.0974

=== Running models for eis ===

[LASSO] Results for eis:
r2: 0.0013
auc: 0.5151
mse: 0.1178

[RIDGE] Results for eis:


In [28]:
#Flatten it into a list of rows
rows = []
for gene, models in all_results.items():
    for model, metrics in models.items():
        row = {'gene': gene, 'model': model}
        row.update(metrics)
        rows.append(row)

In [29]:
# Convert to DataFrame
df = pd.DataFrame(rows)
df

,gene,model,r2,auc,mse
0,rpoB,lasso,0.818375,0.958284,0.032099
1,rpoB,ridge,0.834559,0.962801,0.029239
2,rpoB,logreg,0.962631,0.956946,0.031139
3,rpsL,lasso,0.435267,0.773858,0.120502
4,rpsL,ridge,0.435422,0.773858,0.120469
5,rpsL,logreg,0.773858,0.721115,0.173415
6,tlyA,lasso,-0.000001,0.500000,0.117260
7,tlyA,ridge,0.005227,0.502377,0.116647
8,tlyA,logreg,0.502377,0.505155,0.134266
9,pncA,lasso,0.144806,0.623855,0.094918


In [35]:
# Optional: rearrange columns
cols = ['gene', 'model'] + [c for c in df.columns if c not in ['gene', 'model']]
df = df[cols]

# View result
print(df.head())

   gene   model        r2       auc       mse
0  rpoB   lasso  0.818375  0.958284  0.032099
1  rpoB   ridge  0.834559  0.962801  0.029239
2  rpoB  logreg  0.962631  0.956946  0.031139
3  rpsL   lasso  0.435267  0.773858  0.120502
4  rpsL   ridge  0.435422  0.773858  0.120469


In [36]:
df.to_csv("all_genes_regression_models_performance.csv", index=False)

## precision-recall